In [1]:
import pandas as pd
import numpy as np
import torch
from TRGAN.TRGAN_train_load_modules import embeddings, load_model
from Scripts.data_preprocessing_sber import preprocessing_data_from_sber


In [2]:
data = preprocessing_data_from_sber(folder_path=r'Data\Sber\ditry_single')

Ищем CSV-файлы в: c:\Users\kiril\TRGAN\Data\Sber\ditry_single
Найдено файлов: 1
Файлы: ['c:\\Users\\kiril\\TRGAN\\Data\\Sber\\ditry_single\\transact1_2K.csv']
Загрузка и обработка файла: c:\Users\kiril\TRGAN\Data\Sber\ditry_single\transact1_2K.csv
Корректная обработка ADDRESS...


KeyboardInterrupt: 

In [ ]:
onehot_cols = ['PaymentSystem', 'OpType', 'DETAILEDCARDTYPE', 'TRANMETHOD', 'ISOWNTERMINAL', 'NAME', 'MCC', 'DEVICETYPE', 'CurrencyName', 'CITY', 'REGION', 'ISVIRTUALTRANSACTION']
cat_feat_names = ['ACCOUNT_ID', 'TERMINAL_CODE', 'CARD', 'TRANS_DETAIL']
num_feat_names = ['AMOUNT_EQ', 'HOUR', 'MINUTE', 'SECOND', 'EXCHANGE_RATA']
log1p_transform_cols = ['AMOUNT_EQ', 'EXCHANGE_RATA']  # если суммы имеют skewed распределение
date_feature = 'DATE'
time_feature = 'TRANS_TIME'
client_id = 'ACCOUNT_ID'
mcc_name = 'MCC'
latent_dim = {'onehot': 128, 'categorical': 16, 'numerical': 4, 'cv': 32}

In [ ]:
data["REGION"].unique()

array(['VIRTUAL', 'Санкт-Петербург', 'UNKNOWN', 'обл Ленинградская',
       'обл Новгородская', 'обл Калининградская'], dtype=object)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 275259 entries, 122341 to 2923
Data columns (total 36 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   CustomerKey           275259 non-null  object        
 1   ID                    275259 non-null  object        
 2   TRANS_TIME            275259 non-null  datetime64[ns]
 3   AMOUNT_EQ             275259 non-null  float64       
 4   MCC                   275259 non-null  object        
 5   PAY_AMT               275259 non-null  float64       
 6   CurrencyName          275259 non-null  object        
 7   PaymentSystem         275259 non-null  object        
 8   TERMINAL_CODE         275259 non-null  object        
 9   ADDRESS               275259 non-null  object        
 10  DEVICETYPE            275259 non-null  object        
 11  TRANS_DETAIL          275259 non-null  object        
 12  NAME                  275259 non-null  object        
 13  O

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
directory = 'Pretrains_sber_one_file/'  # папка с сохраненными моделями
experiment_id = 'sber'

In [ ]:
len(data['TERMINAL_CODE'].unique())

52299

In [ ]:
data[data['TERMINAL_CODE'] == 'NONE']

,CustomerKey,ID,TRANS_TIME,AMOUNT_EQ,MCC,PAY_AMT,CurrencyName,PaymentSystem,TERMINAL_CODE,ADDRESS,...,IS_PHYSICAL_LOCATION,ADDRESS_DETAIL_LEVEL,HAS_PHYSICAL_ADDRESS,ISVIRTUALTRANSACTION,OFFLINE_WITH_ADDRESS,ONLINE_VIRTUAL,EXCHANGE_RATA,HOUR,MINUTE,SECOND


In [ ]:
X_emb, X_oh, cond_vector, synth_date, scaler_cat, scaler_onehot, scaler_num, cv_params, scaler, round_array = embeddings(
    data=data,  # можно передать None или заглушку, т.к. load=True
    cat_feat_names=cat_feat_names,
    num_feat_names=num_feat_names,
    onehot_cols=onehot_cols,
    date_feature=date_feature,
    time_feature=time_feature,
    client_id=client_id,
    latent_dim=latent_dim,
    device=device,
    load=True,  # ВАЖНО: загружаем предобученные!
    directory=directory
)

c:\Users\kiril\TRGAN\TRGAN\TRGAN_train_load_modules.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder_onehot.load_state_dict(torch.load(directory + 'onehot_encode

In [ ]:
print(round_array)

[2, 0]


In [ ]:
scaler_num['index_df_sort']

array([[216979,      0,      0,      0, 214781],
       [ 71335,      1,      1,      1, 212997],
       [108331,      2,      2,      2, 213117],
       ...,
       [ 43312, 274943, 275114, 275122, 275256],
       [223033, 274944, 275221, 275132, 275257],
       [150169, 275142, 275255, 275156, 275258]], shape=(275259, 5))

In [ ]:
scaler_num.keys()

dict_keys(['encoder', 'decoder', 'scaler_minmax', 'scaler', 'index_arr', 'index_df_sort'])

In [ ]:
scaler_num["decoder"]

Decoder_cont_emb(
  (model): Sequential(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): ReLU()
    (8): Linear(in_features=128, out_features=5, bias=True)
    (9): Tanh()
  )
)

In [ ]:
# Загружаем обученные GAN модели
generator, supervisor, loss_array = load_model(
    latent_dim=latent_dim,
    dim_noise=20,  # размерность шума (должен совпадать с обучением)
    experiment_id=experiment_id,
    DIRECTORY=directory,
    DEVICE=device,
)

In [ ]:
from TRGAN.TRGAN_main_V2 import sample, inverse_transform

# Сколько образцов сгенерировать
n_samples = 400000  # столько, сколько нужно

# Генерируем синтетические эмбеддинги и даты
synth_data, synth_date_gen, params = sample(
    n_samples=n_samples,
    generator=generator,
    supervisor=supervisor,
    noise_dim=20,  # размерность шума
    cond_vector=cond_vector,
    X_emb=X_emb,
    encoder=cv_params['encoder'],  # энкдер условного вектора
    data=data,  # нужен для генерации времени (можно передать исходные данные или заглушку)
    date_feature=date_feature,
    name_client_id=client_id,
    time_type='synth',  # или 'initial' если хочешь исходные времена
    cv_params=cv_params,
    device=device
)

In [ ]:
synth_date_gen

,DATE
0,2017-05-01
1,2017-05-01
2,2017-05-01
3,2017-05-01
4,2017-05-01
...,...
399995,2026-03-18
399996,2026-03-18
399997,2026-03-18
399998,2026-03-18


In [ ]:
has_nan = np.isnan(synth_data).any()
print(f"Есть ли NaN в массиве: {has_nan}")

Есть ли NaN в массиве: False


In [ ]:
scaler_cat

{'encoder': Encoder_client_emb(
   (model): Sequential(
     (0): Linear(in_features=4, out_features=64, bias=True)
     (1): ReLU()
     (2): Linear(in_features=64, out_features=128, bias=True)
     (3): ReLU()
     (4): Linear(in_features=128, out_features=128, bias=True)
     (5): ReLU()
     (6): Linear(in_features=128, out_features=128, bias=True)
     (7): ReLU()
     (8): Linear(in_features=128, out_features=16, bias=True)
     (9): Tanh()
   )
 ),
 'decoder': Decoder_client_emb(
   (model): Sequential(
     (0): Linear(in_features=16, out_features=64, bias=True)
     (1): ReLU()
     (2): Linear(in_features=64, out_features=128, bias=True)
     (3): ReLU()
     (4): Linear(in_features=128, out_features=128, bias=True)
     (5): ReLU()
     (6): Linear(in_features=128, out_features=128, bias=True)
     (7): ReLU()
     (8): Linear(in_features=128, out_features=4, bias=True)
     (9): Tanh()
   )
 ),
 'scaler': MinMaxScaler(feature_range=(-1, 1)),
 'freq_encoder': [UniformEncoder

In [ ]:
scaler_num["index_df_sort"]

array([[216979,      0,      0,      0, 214781],
       [ 71335,      1,      1,      1, 212997],
       [108331,      2,      2,      2, 213117],
       ...,
       [ 43312, 274943, 275114, 275122, 275256],
       [223033, 274944, 275221, 275132, 275257],
       [150169, 275142, 275255, 275156, 275258]], shape=(275259, 5))

In [ ]:
X_emb1 = scaler.inverse_transform(X_emb)
synth_data = scaler.inverse_transform(synth_data)
print(synth_data)

synth_df = inverse_transform(synth_data, latent_dim, X_oh.columns, scaler_onehot, scaler_cat, scaler_num, cat_feat_names,
                             mcc_name, num_feat_names, True, synth_date_gen, time_feature, round_array, device=device)

[[-0.9432016   0.52882373 -0.02673365 ...  0.98474336  0.7437549
  -0.5283114 ]
 [-0.6756887   0.17749114 -0.13139796 ...  0.98370653  0.7257563
  -0.30552575]
 [-0.68924034  0.9893184  -0.99767417 ...  0.98336047  0.7209697
  -0.5147111 ]
 ...
 [-0.24374264  0.99925214 -0.9846215  ...  0.42588577 -0.55906373
   0.4700189 ]
 [ 0.19535977  0.9849376  -0.9695803  ...  0.38252386 -0.5749911
   0.44484156]
 [-0.5710651   0.99945486 -0.99647254 ...  0.44824395 -0.5618277
   0.46774888]]


In [ ]:
print(latent_dim['numerical'])

4


In [ ]:
synth_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400000 entries, 0 to 399999
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   PaymentSystem         400000 non-null  object        
 1   OpType                400000 non-null  object        
 2   DETAILEDCARDTYPE      400000 non-null  object        
 3   TRANMETHOD            400000 non-null  object        
 4   ISOWNTERMINAL         400000 non-null  object        
 5   NAME                  400000 non-null  object        
 6   MCC                   400000 non-null  object        
 7   DEVICETYPE            400000 non-null  object        
 8   CurrencyName          400000 non-null  object        
 9   CITY                  400000 non-null  object        
 10  REGION                400000 non-null  object        
 11  ISVIRTUALTRANSACTION  400000 non-null  object        
 12  ACCOUNT_ID            400000 non-null  object        
 13 

In [ ]:
synth_df['ISOWNTERMINAL'].unique()

array(['1', '0'], dtype=object)

In [ ]:
synth_df[synth_df['TERMINAL_CODE'] == 'NONE']

,PaymentSystem,OpType,DETAILEDCARDTYPE,TRANMETHOD,ISOWNTERMINAL,NAME,MCC,DEVICETYPE,CurrencyName,CITY,REGION,ISVIRTUALTRANSACTION,ACCOUNT_ID,TERMINAL_CODE,CARD,TRANS_DETAIL,AMOUNT_EQ,EXCHANGE_RATA,DATE,TRANS_TIME


In [ ]:
print("Ключи в scaler_cat:", scaler_cat.keys())

Ключи в scaler_cat: dict_keys(['encoder', 'decoder', 'scaler', 'freq_encoder', 'index_arr'])


In [ ]:
synth_df = synth_df.sample(frac=1, random_state=22)
synth_df.head()

,PaymentSystem,OpType,DETAILEDCARDTYPE,TRANMETHOD,ISOWNTERMINAL,NAME,MCC,DEVICETYPE,CurrencyName,CITY,REGION,ISVIRTUALTRANSACTION,ACCOUNT_ID,TERMINAL_CODE,CARD,TRANS_DETAIL,AMOUNT_EQ,EXCHANGE_RATA,DATE,TRANS_TIME
386986,MasterCard,Оплата,MasterCard World,51,0,Оплата товаров/услуг по карте,5411,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,7039341,20656543,4268763,"MAGAZIN IVA, SOSNOVIY BOR, RU",406.43,0.999794,2018-01-06,16:28:28
398961,МИР - НСПК,Оплата,МИР Классическая Зарплатная,51,0,Оплата товаров/услуг по карте,5200,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,1434794,10533327,3502959,"GRAFOMAN, SHUSHARY, RU",164.27,0.999706,2017-10-01,12:12:12
314025,MasterCard,Оплата,MasterCard World,51,0,Оплата товаров/услуг по карте,5921,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,3979262,10579669,2593106,"RNAZK 6 ""RN-TRADE"", LOMONOSOV, RU",3252.41,1.000000,2017-06-26,23:58:58
386195,МИР - НСПК,Оплата,МИР Классическая Зарплатная,71,0,Оплата товаров/услуг по карте,5499,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,1433952,10726894,3518903,"VERNYI PR. KUL'TURY, ST PETERBURG, RU",173.69,0.999737,2017-10-05,12:14:13
322196,MasterCard,Оплата,MasterCard Unembossed,71,0,Оплата товаров/услуг по карте,5533,VIRTUAL,Рубль,VIRTUAL,VIRTUAL,1,6082409,294443,2347510,"PRODUKTY, KOLPINO, RU",188.20,0.999778,2018-05-30,13:15:15


In [ ]:
synth_df[synth_df['REGION'] != 'VIRTUAL']

,PaymentSystem,OpType,DETAILEDCARDTYPE,TRANMETHOD,ISOWNTERMINAL,NAME,MCC,DEVICETYPE,CurrencyName,CITY,REGION,ISVIRTUALTRANSACTION,ACCOUNT_ID,TERMINAL_CODE,CARD,TRANS_DETAIL,AMOUNT_EQ,EXCHANGE_RATA,DATE,TRANS_TIME
240116,БАНК,Оплата,Детская карта,901,1,Оплата товаров/услуг по карте,5641,POS,Рубль,Санкт-Петербург,Санкт-Петербург,0,4024044,10442457,333787,"SN NORD 10, St Petersburg, RU",272.99,0.999792,2017-05-14,14:21:21
141870,МИР - НСПК,Оплата,МИР Классическая Зарплатная,51,1,Оплата товаров/услуг по карте,5945,POS,Рубль,Петергоф,Санкт-Петербург,0,4189042,AC090187,2269668,"OPTIKOV, SANKT-PETERBU, RU",1679.90,0.999936,2017-06-14,20:52:52
30826,MasterCard,Оплата,MasterCard Unembossed,51,1,Оплата услуг через банкомат,4814,ATM,Рубль,Ломоносов,Санкт-Петербург,0,1837355,482669,350841,"TABAKON, SANKT-PETERBU, RU",2517.05,0.999990,2017-05-05,21:56:56
326673,MasterCard,Снятие наличных,MasterCard Unembossed,51,1,Выдача наличных ден. средств через банкомат,6011,ATM,Рубль,Зеленогорск,обл Ленинградская,0,2130806,10812690,3293099,"IP SADYGOV D M, SANKT-PETERBU, RU",1432.58,0.999892,2017-09-05,19:49:49
271099,MasterCard,Снятие наличных,MasterCard Unembossed,51,1,Выдача наличных ден. средств через банкомат,6011,ATM,Рубль,Санкт-Петербург,Санкт-Петербург,0,11352593,11496499,3979533,"MAGNIT MM DUBROVKA, KIRISHI, RU",919.62,0.999816,2017-11-28,18:43:43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106077,MasterCard,Оплата,MasterCard World,51,1,Оплата услуг через банкомат,4814,ATM,Рубль,Санкт-Петербург,Санкт-Петербург,0,128758,70001018,3368888,"VERNYI UL ANTONOVA O, ST PETERSBURG, RU",180.80,0.999276,2017-09-10,11:10:10
101436,МИР - НСПК,Оплата,МИР Классическая Зарплатная,51,1,Оплата товаров/услуг по карте,5462,POS,Рубль,Санкт-Петербург,Санкт-Петербург,0,1433926,11436154,3706664,"OOO KVAZAR, SANKT-PETERBU, RU",1401.99,0.999932,2017-10-27,20:50:50
223997,MasterCard,Оплата,MasterCard Gold,51,1,Оплата услуг через банкомат,4814,ATM,Рубль,Санкт-Петербург,Санкт-Петербург,0,3443305,872760,2911030,"ZAGORODNIY, 52A, St Petersburg, RU",525.87,0.999804,2019-08-06,17:33:33
237177,МИР - НСПК,Снятие наличных,МИР Классическая Зарплатная,51,1,Выдача наличных ден. средств через банкомат,6011,ATM,Рубль,Санкт-Петербург,Санкт-Петербург,0,5037591,11352409,3498273,"PYATEROCHKA 2418, ROZHDESTVENO, RU",1826.69,0.999962,2017-10-11,20:53:53


In [ ]:
synth_df.to_csv('Data/Sber/Clear/transact_400000_samples.csv', 
          index=False,           # Не записывать индексы
          sep=',',               # Разделитель
          encoding='utf-8',      # Кодировка
          header=True,           # Записывать заголовки
          na_rep='NULL')         # Замена NaN значений